# Instacart Basket Creation

This notebook builds basket tables from order lines and prepares baskets for Apriori.

## Goals
- Build basket tables for train_rules, validation, and test_final
- Keep customer segment labels
- Select a reduced set of products for Apriori (proportional by department)
- Filter and sample baskets for rule mining
- Save all basket outputs for the next notebooks

## Output
This notebook saves basket tables and product selection diagnostics in the outputs folder.

In [3]:
import os
import pandas as pd
import numpy as np

In [4]:
def load_instacart_tables():
    """
    Load the main Instacart tables.
    """
    tables = {
        "orders": pd.read_csv("../data/orders.csv"),
        "order_products__prior": pd.read_csv("../data/order_products__prior.csv"),
        "order_products__train": pd.read_csv("../data/order_products__train.csv"),
        "products": pd.read_csv("../data/products.csv"),
        "aisles": pd.read_csv("../data/aisles.csv"),
        "departments": pd.read_csv("../data/departments.csv"),
    }
    return tables


def load_split_orders():
    """
    Load split order tables from Notebook 3.
    """
    tables = {
        "train_rules": pd.read_csv("../outputs/orders_train_rules.csv"),
        "validation": pd.read_csv("../outputs/orders_validation.csv"),
        "test_final": pd.read_csv("../outputs/orders_test_final.csv"),
    }
    return tables


def enrich_products(products, aisles, departments):
    """
    Merge products with aisle and department names.
    """
    df = products.merge(aisles, on="aisle_id", how="left")
    df = df.merge(departments, on="department_id", how="left")
    return df


def build_order_products_all(op_prior, op_train):
    """
    Combine prior and train order lines in one table.
    """
    df = pd.concat([op_prior, op_train], ignore_index=True)
    return df


def build_baskets(order_lines, split_orders):
    """
    Build baskets with one row per order and a list of product IDs.
    """
    base_cols = [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "segment",
    ]

    split_base = split_orders[base_cols].copy()

    items_df = order_lines[order_lines["order_id"].isin(split_base["order_id"])].copy()

    basket_items = items_df.groupby("order_id")["product_id"].apply(list).reset_index(name="items")

    baskets = split_base.merge(basket_items, on="order_id", how="left")
    baskets["items"] = baskets["items"].apply(lambda x: x if isinstance(x, list) else [])

    baskets["basket_size"] = baskets["items"].apply(len)
    return baskets


def basket_overview(baskets_df, name):
    """
    Build a simple basket summary.
    """
    row = {
        "dataset": name,
        "n_orders": len(baskets_df),
        "n_users": baskets_df["user_id"].nunique(),
        "avg_basket_size": baskets_df["basket_size"].mean(),
        "median_basket_size": baskets_df["basket_size"].median(),
        "min_basket_size": baskets_df["basket_size"].min(),
        "max_basket_size": baskets_df["basket_size"].max(),
        "empty_baskets": int((baskets_df["basket_size"] == 0).sum()),
    }
    return pd.DataFrame([row])


def product_occurrences_from_baskets(baskets_df):
    """
    Count product occurrences from a basket table.
    """
    exploded = baskets_df[["order_id", "items"]].explode("items").rename(columns={"items": "product_id"})
    exploded = exploded.dropna(subset=["product_id"]).copy()
    exploded["product_id"] = exploded["product_id"].astype(int)

    occ = exploded.groupby("product_id").size().reset_index(name="n_occurrences")
    occ = occ.sort_values("n_occurrences", ascending=False).reset_index(drop=True)
    return occ


def attach_product_metadata(product_occ, products_enriched):
    """
    Add product, aisle, and department names to product counts.
    """
    df = product_occ.merge(
        products_enriched[["product_id", "product_name", "department", "aisle"]],
        on="product_id",
        how="left",
    )
    return df


def allocate_department_quotas(product_occ_enriched, target_n_products=3000):
    """
    Allocate product quotas per department using occurrence shares.
    """
    dep = product_occ_enriched.groupby("department").agg(
        dept_occurrences=("n_occurrences", "sum"),
        n_available_products=("product_id", "nunique"),
    ).reset_index()

    dep = dep[dep["department"].notna()].copy()

    dep["occ_share"] = dep["dept_occurrences"] / dep["dept_occurrences"].sum()
    dep["quota_raw"] = dep["occ_share"] * target_n_products
    dep["quota_floor"] = np.floor(dep["quota_raw"]).astype(int)
    dep["quota_remainder"] = dep["quota_raw"] - dep["quota_floor"]

    dep["quota"] = dep[["quota_floor", "n_available_products"]].min(axis=1)

    current_total = int(dep["quota"].sum())
    remaining = int(target_n_products - current_total)

    if remaining > 0:
        dep = dep.sort_values("quota_remainder", ascending=False).reset_index(drop=True)

        while remaining > 0:
            changed = False
            for i in dep.index:
                if dep.loc[i, "quota"] < dep.loc[i, "n_available_products"]:
                    dep.loc[i, "quota"] += 1
                    remaining -= 1
                    changed = True
                    if remaining == 0:
                        break
            if not changed:
                break

    dep = dep.sort_values("dept_occurrences", ascending=False).reset_index(drop=True)
    return dep


def select_products_by_department_quota(product_occ_enriched, dep_quota):
    """
    Select top products inside each department quota.
    """
    df = product_occ_enriched.copy()
    df = df[df["department"].notna()].copy()

    df["rank_in_department"] = df.groupby("department")["n_occurrences"].rank(
        method="first",
        ascending=False,
    )

    df = df.merge(dep_quota[["department", "quota", "occ_share"]], on="department", how="left")

    selected = df[df["rank_in_department"] <= df["quota"]].copy()
    selected = selected.sort_values(["department", "rank_in_department"]).reset_index(drop=True)

    selected_ids = set(selected["product_id"].tolist())
    return selected_ids, selected


def build_department_distribution_report(baskets_df, selected_ids, products_enriched):
    """
    Compare department shares before and after product selection.
    """
    exploded = baskets_df[["order_id", "items"]].explode("items").rename(columns={"items": "product_id"})
    exploded = exploded.dropna(subset=["product_id"]).copy()
    exploded["product_id"] = exploded["product_id"].astype(int)

    exploded = exploded.merge(
        products_enriched[["product_id", "department"]],
        on="product_id",
        how="left",
    )

    before = exploded.groupby("department").size().reset_index(name="occ_before")
    before["share_before"] = before["occ_before"] / before["occ_before"].sum()

    after_data = exploded[exploded["product_id"].isin(selected_ids)].copy()
    after = after_data.groupby("department").size().reset_index(name="occ_after")
    after["share_after"] = after["occ_after"] / after["occ_after"].sum()

    report = before.merge(after, on="department", how="outer").fillna(0)

    selected_count = (
        after_data[["product_id", "department"]]
        .drop_duplicates()
        .groupby("department")
        .size()
        .reset_index(name="n_selected_products")
    )
    report = report.merge(selected_count, on="department", how="outer").fillna(0)

    report["occ_before"] = report["occ_before"].astype(int)
    report["occ_after"] = report["occ_after"].astype(int)
    report["n_selected_products"] = report["n_selected_products"].astype(int)
    report["share_diff_abs"] = (report["share_after"] - report["share_before"]).abs()

    report = report.sort_values("share_before", ascending=False).reset_index(drop=True)
    return report


def filter_baskets_to_selected_products(baskets_df, selected_ids):
    """
    Keep only selected products inside each basket.
    """
    df = baskets_df.copy()
    df["items"] = df["items"].apply(lambda items: [p for p in items if p in selected_ids])
    df["basket_size"] = df["items"].apply(len)
    return df


def keep_baskets_with_min_items(baskets_df, min_items=2):
    """
    Keep baskets with at least min_items products.
    """
    df = baskets_df[baskets_df["basket_size"] >= min_items].copy()
    return df


def sample_baskets_by_segment(baskets_df, sample_n=150000, random_state=42):
    """
    Sample baskets proportionally by segment.
    """
    df = baskets_df.copy()

    if sample_n is None or len(df) <= sample_n:
        return df

    if "segment" not in df.columns:
        return df.sample(n=sample_n, random_state=random_state).copy()

    shares = df["segment"].value_counts(normalize=True)
    parts = []

    for seg, share in shares.items():
        seg_df = df[df["segment"] == seg]
        n_seg = int(round(sample_n * share))
        n_seg = min(n_seg, len(seg_df))
        if n_seg > 0:
            parts.append(seg_df.sample(n=n_seg, random_state=random_state))

    sampled = pd.concat(parts, ignore_index=False).drop_duplicates(subset=["order_id"])

    missing = sample_n - len(sampled)
    if missing > 0:
        remaining = df[~df["order_id"].isin(sampled["order_id"])]
        if len(remaining) > 0:
            extra = remaining.sample(n=min(missing, len(remaining)), random_state=random_state)
            sampled = pd.concat([sampled, extra], ignore_index=False)

    sampled = sampled.reset_index(drop=True)
    return sampled


def prep_summary_table(
    baskets_train_rules,
    baskets_train_rules_ready,
    selected_products,
    dep_report,
    target_n_products,
    sample_n_baskets,
):
    """
    Build a summary table for Apriori basket preparation.
    """
    row = {
        "selection_method": "proportional_by_department_occurrences",
        "target_n_products": target_n_products,
        "n_products_selected": len(selected_products["product_id"].unique()),
        "sample_n_baskets_target": sample_n_baskets,
        "n_baskets_before_filter": len(baskets_train_rules),
        "n_baskets_after_filter": len(baskets_train_rules_ready),
        "avg_basket_size_after_filter": baskets_train_rules_ready["basket_size"].mean(),
        "avg_share_diff_abs_departments": dep_report["share_diff_abs"].mean(),
    }
    return pd.DataFrame([row])


def save_baskets_outputs(
    baskets_all,
    baskets_train_rules,
    baskets_validation,
    baskets_test,
    baskets_train_rules_ready,
    selected_products,
    dep_quota,
    dep_report,
    prep_summary,
):
    """
    Save basket and selection outputs.
    """
    os.makedirs("../outputs", exist_ok=True)

    baskets_all.to_csv("../outputs/baskets_all.csv", index=False)
    baskets_train_rules.to_csv("../outputs/baskets_train_rules.csv", index=False)
    baskets_validation.to_csv("../outputs/baskets_validation.csv", index=False)
    baskets_test.to_csv("../outputs/baskets_test.csv", index=False)

    baskets_train_rules_ready.to_csv("../outputs/baskets_train_rules_ready.csv", index=False)

    selected_products.to_csv("../outputs/apriori_train_rules_selected_products.csv", index=False)
    dep_quota.to_csv("../outputs/apriori_train_rules_department_quotas.csv", index=False)
    dep_report.to_csv("../outputs/apriori_train_rules_department_distribution_before_after.csv", index=False)
    prep_summary.to_csv("../outputs/apriori_train_rules_prep_summary.csv", index=False)

In [5]:
# Load data
tables = load_instacart_tables()
split_tables = load_split_orders()

orders = tables["orders"]
op_prior = tables["order_products__prior"]
op_train = tables["order_products__train"]
products = tables["products"]
aisles = tables["aisles"]
departments = tables["departments"]

orders_train_rules = split_tables["train_rules"]
orders_validation = split_tables["validation"]
orders_test_final = split_tables["test_final"]

In [6]:
# Build product metadata
products_enriched = enrich_products(products, aisles, departments)

# Build combined order lines
order_lines_all = build_order_products_all(op_prior, op_train)
display(order_lines_all.head())

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [7]:
# Build baskets for each split
baskets_train_rules = build_baskets(order_lines_all, orders_train_rules)
baskets_validation = build_baskets(order_lines_all, orders_validation)
baskets_test = build_baskets(order_lines_all, orders_test_final)

# Build one full basket table (all split baskets together)
baskets_all = pd.concat(
    [baskets_train_rules, baskets_validation, baskets_test],
    ignore_index=True,
)

# Show basket overviews
display(basket_overview(baskets_train_rules, "train_rules"))
display(basket_overview(baskets_validation, "validation"))
display(basket_overview(baskets_test, "test_final"))
display(basket_overview(baskets_all, "all"))

,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,train_rules,3008665,206209,10.069151,8.0,1,145,0


,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,validation,206209,206209,10.376792,9.0,1,121,0


,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,test_final,131209,131209,10.552759,9.0,1,80,0


,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,all,3346083,206209,10.107073,8.0,1,145,0


In [8]:
# Show segment distribution in basket tables
seg_train = baskets_train_rules["segment"].value_counts(normalize=True).reset_index()
seg_train.columns = ["segment", "share_train_rules"]
display(seg_train)

seg_val = baskets_validation["segment"].value_counts(normalize=True).reset_index()
seg_val.columns = ["segment", "share_validation"]
display(seg_val)

seg_test = baskets_test["segment"].value_counts(normalize=True).reset_index()
seg_test.columns = ["segment", "share_test_final"]
display(seg_test)

,segment,share_train_rules
0,heavy,0.685462
1,frequent,0.213074
2,rare,0.101464


,segment,share_validation
0,heavy,0.338302
1,rare,0.331072
2,frequent,0.330626


,segment,share_test_final
0,heavy,0.338788
1,rare,0.331212
2,frequent,0.330000


In [9]:
# Build product occurrence table on train_rules only
product_occ = product_occurrences_from_baskets(baskets_train_rules)
product_occ_enriched = attach_product_metadata(product_occ, products_enriched)

display(product_occ_enriched.head(20))
display(product_occ_enriched.tail(20))

,product_id,n_occurrences,product_name,department,aisle
0,24852,443151,Banana,produce,fresh fruits
1,13176,355357,Bag of Organic Bananas,produce,fresh fruits
2,21137,249093,Organic Strawberries,produce,fresh fruits
3,21903,226511,Organic Baby Spinach,produce,packaged vegetables fruits
4,47209,202023,Organic Hass Avocado,produce,fresh fruits
5,47766,165270,Organic Avocado,produce,fresh fruits
6,47626,140557,Large Lemon,produce,fresh fruits
7,16797,132939,Strawberries,produce,fresh fruits
8,26209,131440,Limes,produce,fresh fruits
9,27845,130239,Organic Whole Milk,dairy eggs,milk


,product_id,n_occurrences,product_name,department,aisle
49632,24402,1,Orangemint Flavored Water,beverages,water seltzer sparkling water
49633,44982,1,Coconut Bliss Pineapple Coconut,frozen,ice cream ice
49634,44986,1,Mustard & Onion,snacks,chips pretzels
49635,21396,1,Citrus Vodka,alcohol,spirits
49636,21264,1,2 Flavors in One Crazy Beans,missing,missing
49637,24348,1,Salted Caramel Craze Ice Cream,missing,missing
49638,36998,1,Microwavable Triple Cheese Macaroni & Cheese,dry goods pasta,instant foods
49639,32021,1,Ginseng Vitality Tea,beverages,tea
49640,5081,1,Deep Clean Cleanser & Mask,personal care,facial care
49641,10311,1,Brownie Mix 13 x 9 Family Size Dark Chocolate,pantry,doughs gelatins bake mixes


In [10]:
# Select products proportionally by department
target_n_products = 3000

dep_quota = allocate_department_quotas(
    product_occ_enriched,
    target_n_products=target_n_products,
)
display(dep_quota)

selected_ids, selected_products = select_products_by_department_quota(
    product_occ_enriched,
    dep_quota,
)
display(selected_products.head(30))

,department,dept_occurrences,n_available_products,occ_share,quota_raw,quota_floor,quota_remainder,quota
0,produce,8854642,1684,0.292284,876.850575,876,0.850575,877
1,dairy eggs,5077769,3447,0.167612,502.837344,502,0.837344,503
2,snacks,2703101,6260,0.089227,267.680576,267,0.680576,268
3,beverages,2513695,4362,0.082975,248.924226,248,0.924226,249
4,frozen,2080442,4007,0.068673,206.020386,206,0.020386,206
5,pantry,1749048,5368,0.057734,173.203360,173,0.203360,173
6,bakery,1101488,1515,0.036359,109.077294,109,0.077294,109
7,canned goods,993053,2091,0.032780,98.339277,98,0.339277,98
8,deli,982706,1322,0.032438,97.314643,97,0.314643,97
9,dry goods pasta,806130,1858,0.026610,79.828812,79,0.828812,80


,product_id,n_occurrences,product_name,department,aisle,rank_in_department,quota,occ_share
0,2120,7737,Sauvignon Blanc,alcohol,white wines,1.0,14,0.004752
1,38444,5795,Chardonnay,alcohol,white wines,2.0,14,0.004752
2,33065,5704,Cabernet Sauvignon,alcohol,red wines,3.0,14,0.004752
3,46088,5489,Beer,alcohol,beers coolers,4.0,14,0.004752
4,45190,5157,Vodka,alcohol,spirits,5.0,14,0.004752
5,41131,3997,India Pale Ale,alcohol,beers coolers,6.0,14,0.004752
6,12013,3715,Pinot Noir,alcohol,red wines,7.0,14,0.004752
7,1160,3053,Pinot Grigio,alcohol,white wines,8.0,14,0.004752
8,27885,2272,Malbec,alcohol,red wines,9.0,14,0.004752
9,36425,1848,Chardonnay Wine,alcohol,white wines,10.0,14,0.004752


In [11]:
# Compare department distribution before/after selection
dep_report = build_department_distribution_report(
    baskets_train_rules,
    selected_ids,
    products_enriched,
)
display(dep_report)

,department,occ_before,share_before,occ_after,share_after,n_selected_products,share_diff_abs
0,produce,8854642,0.292284,8796817,0.406290,877,0.114007
1,dairy eggs,5077769,0.167612,4124581,0.190498,503,0.022886
2,snacks,2703101,0.089227,1361698,0.062891,268,0.026335
3,beverages,2513695,0.082975,1616072,0.074640,249,0.008335
4,frozen,2080442,0.068673,1133581,0.052356,206,0.016318
5,pantry,1749048,0.057734,865173,0.039959,173,0.017776
6,bakery,1101488,0.036359,644614,0.029772,109,0.006587
7,canned goods,993053,0.032780,588813,0.027195,98,0.005585
8,deli,982706,0.032438,656081,0.030302,97,0.002136
9,dry goods pasta,806130,0.026610,420816,0.019436,80,0.007174


In [12]:
# Prepare baskets for Apriori on train_rules
baskets_train_rules_filtered = filter_baskets_to_selected_products(
    baskets_train_rules,
    selected_ids,
)

baskets_train_rules_filtered = keep_baskets_with_min_items(
    baskets_train_rules_filtered,
    min_items=2,
)

sample_n_baskets = 150000

baskets_train_rules_ready = sample_baskets_by_segment(
    baskets_train_rules_filtered,
    sample_n=sample_n_baskets,
    random_state=42,
)

display(basket_overview(baskets_train_rules_filtered, "train_rules_filtered"))
display(basket_overview(baskets_train_rules_ready, "train_rules_ready"))

,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,train_rules_filtered,2651897,201924,8.067704,7.0,2,108,0


,dataset,n_orders,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size,empty_baskets
0,train_rules_ready,150001,83209,8.065,7.0,2,75,0


In [13]:
# Build preparation summary
prep_summary = prep_summary_table(
    baskets_train_rules=baskets_train_rules,
    baskets_train_rules_ready=baskets_train_rules_ready,
    selected_products=selected_products,
    dep_report=dep_report,
    target_n_products=target_n_products,
    sample_n_baskets=sample_n_baskets,
)
display(prep_summary)

# Show selected product distribution by department
selected_dep = selected_products.groupby("department").agg(
    n_selected_products=("product_id", "nunique"),
    total_occurrences=("n_occurrences", "sum"),
).reset_index()

selected_dep = selected_dep.sort_values("total_occurrences", ascending=False).reset_index(drop=True)
display(selected_dep)

,selection_method,target_n_products,n_products_selected,sample_n_baskets_target,n_baskets_before_filter,n_baskets_after_filter,avg_basket_size_after_filter,avg_share_diff_abs_departments
0,proportional_by_department_occurrences,3000,3000,150000,3008665,150001,8.065,0.013037


,department,n_selected_products,total_occurrences
0,produce,877,8796817
1,dairy eggs,503,4124581
2,beverages,249,1616072
3,snacks,268,1361698
4,frozen,206,1133581
5,pantry,173,865173
6,deli,97,656081
7,bakery,109,644614
8,canned goods,98,588813
9,meat seafood,65,458156


In [14]:
# Save outputs
save_baskets_outputs(
    baskets_all=baskets_all,
    baskets_train_rules=baskets_train_rules,
    baskets_validation=baskets_validation,
    baskets_test=baskets_test,
    baskets_train_rules_ready=baskets_train_rules_ready,
    selected_products=selected_products,
    dep_quota=dep_quota,
    dep_report=dep_report,
    prep_summary=prep_summary,
)

In [15]:
# summary
final_summary = pd.DataFrame([{
    "n_baskets_train_rules": len(baskets_train_rules),
    "n_baskets_validation": len(baskets_validation),
    "n_baskets_test_final": len(baskets_test),
    "n_baskets_train_rules_ready": len(baskets_train_rules_ready),
    "n_selected_products": len(selected_ids),
    "outputs_saved": True,
}])
display(final_summary)

,n_baskets_train_rules,n_baskets_validation,n_baskets_test_final,n_baskets_train_rules_ready,n_selected_products,outputs_saved
0,3008665,206209,131209,150001,3000,True
